# Road Collision Analysis (2025) — Complete Google Colab Project

**Start with the first code cell.** Click **Choose Files** and select your **labelled road collision CSV** (the version where category codes have been replaced by words). No data is embedded and no filename search is performed.

The notebook includes data-quality checks, charts, your Decision Tree model, a confusion matrix, a tree diagram and an optional CarApp website. Run the cells **in order**. The CarApp steps are at the end and can be skipped until needed.

**Note:** Colab’s file uploader is a Google Colab interface feature. If it remains stuck on *Loading*, stop that cell and upload the CSV through Colab’s left-hand **Files** panel instead; the Colab uploader itself cannot be repaired by changing Python code.
**Display fix:** All dashboard KPI cards use dark backgrounds with white text, and charts have explicit high-contrast colours.


In [ ]:
# 1. Upload YOUR labelled CSV using the Choose Files button
import io
import pandas as pd
from pathlib import Path
from google.colab import files

print("Choose your labelled 2025 collision CSV:")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file was selected. Please run this cell again.")

filename, file_bytes = next(iter(uploaded.items()))
if not filename.lower().endswith('.csv'):
    raise ValueError("Please choose a CSV file.")

# Save this file for CarApp to use later in the same Colab session
csv_path = Path('/content/road_collision_uploaded.csv')
csv_path.write_bytes(file_bytes)

df = pd.read_csv(io.BytesIO(file_bytes), low_memory=False)
print("Loaded:", filename)
print("Rows:", len(df), "| Columns:", len(df.columns))
display(df.head())

# Check that this is the labelled version used by the model
expected = {'Slight', 'Serious', 'Fatal'}
if 'collision_severity' not in df.columns:
    raise ValueError("This does not look like the collision dataset: collision_severity is missing.")
if not df['collision_severity'].dropna().isin(expected).all():
    raise ValueError("Please upload the LABELLED CSV with Slight, Serious and Fatal as words.")
print("Ready for your analysis and Decision Tree.")

## 2. Preprocessing and data-quality checks

We examine missing values and exact duplicates. The DfT guide uses descriptive labels for categorical codes. Numeric measurements such as speed limits remain numeric; some unknown values are retained in their original categories rather than being invented.

In [ ]:
# 2. Preprocessing checks
print('Dataset size:', df.shape)
print('Exact duplicate records:', df.duplicated().sum())
missing = df.isna().sum()
print('\nColumns with missing values:')
display(missing[missing > 0].sort_values(ascending=False).to_frame('Missing records'))

print('\nRecorded severity categories:')
display(df['collision_severity'].value_counts().rename_axis('Severity').to_frame('Collisions'))

# Make an unmodified copy for any further exploration
df_clean = df.copy()

## 3. Exploratory data analysis

We compare the distributions of slight, serious and fatal *recorded injury collisions* by environmental conditions. These counts cannot establish the risk of having an accident because traffic exposure varies by weather, lighting, time, and road type.

In [ ]:
# 3A. Severity distribution
import matplotlib.pyplot as plt

severity_counts = df_clean['collision_severity'].value_counts().reindex(
    ['Slight', 'Serious', 'Fatal'], fill_value=0
)
fig, ax = plt.subplots(figsize=(8, 4.5))
severity_counts.plot(kind='bar', ax=ax)
ax.set_title('2025 recorded collisions by severity')
ax.set_xlabel('Severity')
ax.set_ylabel('Collisions')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(severity_counts.values):
    ax.text(i, v + 600, f'{v:,}', ha='center', fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
# 3B. Serious/fatal collision proportions by weather condition
weather = pd.crosstab(df_clean['weather_conditions'], df_clean['collision_severity'])
weather['Total'] = weather.sum(axis=1)
weather['Serious/Fatal'] = weather.get('Serious', 0) + weather.get('Fatal', 0)
weather['Serious/Fatal (%)'] = (100 * weather['Serious/Fatal'] / weather['Total']).round(2)
weather = weather.sort_values('Total', ascending=False)
display(weather[['Total','Serious/Fatal','Serious/Fatal (%)']])

# Plot percentages for weather categories with >=100 collisions
chart = weather[weather['Total'] >= 100]['Serious/Fatal (%)'].sort_values()
fig, ax = plt.subplots(figsize=(11, 5))
chart.plot(kind='barh', ax=ax)
ax.set_title('Serious/fatal share among recorded collisions, by weather')
ax.set_xlabel('Serious or fatal collisions within category (%)')
fig.tight_layout()
plt.show()

In [ ]:
# 3C. Fatal collisions by lighting and serious collisions by road surface
fig, axes = plt.subplots(1, 2, figsize=(17, 5.5))
lighting_fatal = (df_clean.loc[df_clean['collision_severity'].eq('Fatal'), 'light_conditions']
                  .value_counts().sort_values(ascending=True))
lighting_fatal.plot(kind='barh', ax=axes[0])
axes[0].set_title('Fatal collisions by lighting condition')
axes[0].set_xlabel('Fatal collisions')

surface_serious = (df_clean.loc[df_clean['collision_severity'].eq('Serious'), 'road_surface_conditions']
                   .value_counts().sort_values(ascending=True))
surface_serious.plot(kind='barh', ax=axes[1])
axes[1].set_title('Serious collisions by road surface')
axes[1].set_xlabel('Serious collisions')
fig.tight_layout()
plt.show()

print('Fatal collisions by lighting:')
display(lighting_fatal.sort_values(ascending=False).rename('Collisions').to_frame())
print('Serious collisions by road surface:')
display(surface_serious.sort_values(ascending=False).rename('Collisions').to_frame())

## 4. Decision Tree classification

The model predicts **Slight (0)** versus **Serious or Fatal (1)** for *recorded collisions*. It uses six road/environmental variables, a **stratified 80/20 train/test split**, **One-Hot Encoding** for descriptive text categories, and a depth-limited Decision Tree with balanced class weights. This keeps the analysis consistent with the results in your report.

We measure accuracy, class-wise precision/recall/F1, and use a confusion matrix to show errors. The model is **not suitable for individual safety decisions**.

In [ ]:
# 4A. Prepare data and train the Decision Tree with WORD labels
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

features = [
    'speed_limit', 'road_type', 'light_conditions',
    'weather_conditions', 'road_surface_conditions', 'urban_or_rural_area'
]
categorical_features = [name for name in features if name != 'speed_limit']

X = df_clean[features].copy()
X['speed_limit'] = pd.to_numeric(X['speed_limit'], errors='raise')
y = df_clean['collision_severity'].isin(['Serious', 'Fatal']).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer([
    ('categories', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('speed', 'passthrough', ['speed_limit'])
])
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=100,
        class_weight='balanced', random_state=42
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('✅ Decision Tree trained successfully')
print(f'Training records: {len(X_train):,}; Testing records: {len(X_test):,}')
print(f'Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print('\nClassification report:')
print(classification_report(y_test, y_pred,
                            target_names=['Slight', 'Serious/Fatal'], zero_division=0))
print('Confusion matrix [actual rows / predicted columns]:')
print(confusion_matrix(y_test, y_pred))

In [ ]:
# 4B. Confusion matrix and simplified Decision Tree diagram
from sklearn.tree import plot_tree

with plt.style.context('default'):
    fig, ax = plt.subplots(figsize=(7, 5), facecolor='white')
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Slight', 'Serious/Fatal'])
    disp.plot(cmap='Blues', values_format='d', ax=ax, colorbar=False)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            disp.text_[i, j].set_color(
                'white' if cm[i, j] >= cm.max() * 0.55 else '#142c48'
            )
            disp.text_[i, j].set_fontweight('bold')
    ax.tick_params(colors='#142c48')
    ax.set_title('Decision Tree: confusion matrix', color='#142c48')
    fig.tight_layout()
    plt.show()

feature_names = model.named_steps['preprocessor'].get_feature_names_out()
feature_names = [name.replace('categories__', '').replace('speed__', '') for name in feature_names]
with plt.style.context('default'):
    fig, ax = plt.subplots(figsize=(24, 11), facecolor='white')
    plot_tree(model.named_steps['classifier'], feature_names=feature_names,
              class_names=['Slight', 'Serious/Fatal'],
              max_depth=3, filled=True, rounded=True, fontsize=8, ax=ax)
    ax.set_title('Decision Tree: first 3 levels shown (trained max depth = 5)', color='#142c48')
    fig.tight_layout()
    plt.show()

## 5. CarApp — optional website

If you want to demonstrate your analysis in Streamlit, run the following cells in order. The first cell has already saved your selected labelled CSV under `/content/road_collision_uploaded.csv`, which the website uses. The website also offers a file upload in its sidebar if the CSV is missing. No separate application ZIP, embedded dataset, or code conversion is needed.

In [ ]:
# 5A. Install Streamlit
!pip -q install streamlit pandas matplotlib plotly scikit-learn

In [ ]:
%%writefile /content/CarApp.py
"""CarApp — 2025 Great Britain road collision analysis, labelled dataset.

Works directly with the descriptive values in the user's Department for
Transport 2025 collision CSV. Run: streamlit run CarApp.py
"""
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import streamlit as st
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree

st.set_page_config(page_title='CarApp | 2025 Road Collisions', page_icon='🚗', layout='wide')
ROOT = Path(__file__).parent
DATA = Path('/content/road_collision_uploaded.csv')
FEATURES = ['speed_limit', 'road_type', 'light_conditions',
            'weather_conditions', 'road_surface_conditions', 'urban_or_rural_area']
CATEGORICAL = [f for f in FEATURES if f != 'speed_limit']
PLOTS = {
    'Weather conditions': 'weather_conditions',
    'Lighting conditions': 'light_conditions',
    'Road surface': 'road_surface_conditions',
    'Road type': 'road_type',
    'Urban or rural': 'urban_or_rural_area',
    'Speed limit': 'speed_limit',
}
SEVERITIES = ['Slight', 'Serious', 'Fatal']

st.markdown('''
<style>
.block-container {padding-top: 1.3rem; max-width: 1280px;}
/* Keep all native metric cards legible in dark and light Streamlit themes. */
[data-testid="stMetric"] {
  background-color: #16384F !important; border: 1px solid #35657B !important;
  padding: 16px; border-radius: 12px;
}
[data-testid="stMetric"] *, [data-testid="stMetric"] {
  color: #FFFFFF !important;
  -webkit-text-fill-color: #FFFFFF !important;
}

.hero {background: linear-gradient(120deg, #142c48, #147580); border-radius: 16px;
  padding: 26px 30px; color: #fff; margin-bottom: 20px;}
.hero h1 {color: #fff; margin: 0; font-size: 34px;}
.hero p {color: #e2f0f1; margin: 8px 0 0 0;}
</style>
<div class="hero"><h1>🚗 CarApp — Road Collision Analysis</h1>
<p>Great Britain, 2025 · Data exploration and Decision Tree classification</p></div>
''', unsafe_allow_html=True)

@st.cache_data(show_spinner='Loading the 2025 collision dataset…')
def load_data():
    return pd.read_csv(DATA, low_memory=False)

@st.cache_resource(show_spinner='Training Decision Tree using labelled categories…')
def train_tree():
    df = load_data()
    required = FEATURES + ['collision_severity']
    missing = [col for col in required if col not in df]
    if missing:
        raise ValueError(f'Missing dataset columns: {missing}')
    valid = df['collision_severity'].isin(SEVERITIES)
    X = df.loc[valid, FEATURES].copy()
    X['speed_limit'] = pd.to_numeric(X['speed_limit'], errors='raise')
    y = df.loc[valid, 'collision_severity'].isin(['Serious', 'Fatal']).astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    preprocessor = ColumnTransformer([
        ('categories', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL),
        ('speed', 'passthrough', ['speed_limit']),
    ])
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(
            max_depth=5, min_samples_leaf=100,
            class_weight='balanced', random_state=42,
        )),
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return model, y_test, pred, len(X_train), len(X_test)

if not DATA.exists():
    st.warning('Please upload your labelled CSV in the sidebar, or run the first Colab cell.')
    uploaded_csv = st.sidebar.file_uploader('Choose labelled CSV', type=['csv'])
    if uploaded_csv is None:
        st.stop()
    DATA.write_bytes(uploaded_csv.getvalue())
    st.rerun()

try:
    df = load_data()
except Exception as error:
    st.error(f'Could not read the included dataset: {error}')
    st.stop()

st.sidebar.header('Explore the records')
st.sidebar.success('✅ Labelled CSV loaded.')
st.sidebar.caption(f'{len(df):,} police-reported injury collision records (2025).')
st.sidebar.divider()
st.sidebar.markdown('**Source:** Department for Transport, 2025 road safety open data and its supplied data guide.')

levels = df['collision_severity'].value_counts()
# High-contrast KPI cards: dark background and explicit WHITE text.
# These use HTML instead of Streamlit st.metric to avoid theme-dependent colours.
metric_cols = st.columns(4)
for col, title, value in zip(
    metric_cols,
    ['Total collisions', 'Slight', 'Serious', 'Fatal'],
    [len(df), int(levels.get('Slight', 0)),
     int(levels.get('Serious', 0)), int(levels.get('Fatal', 0))]
):
    with col:
        st.markdown(
            f'''<div style="background-color:#16384F; border:1px solid #35657B;
                 border-radius:12px; padding:14px 18px; min-height:108px;">
                <div style="font-size:14px; color:#D9EEF8 !important;
                     -webkit-text-fill-color:#D9EEF8;">{title}</div>
                <div style="font-size:32px; font-weight:750; line-height:1.5;
                     color:#FFFFFF !important; -webkit-text-fill-color:#FFFFFF;">
                    {value:,}
                </div>
            </div>''',
            unsafe_allow_html=True,
        )

summary, exploration, quality, modelling = st.tabs([
    '📊 Overview', '🔎 Collision Analysis', '🧹 Data Preprocessing', '🌳 Decision Tree'
])

with summary:
    a, b = st.columns(2)
    with a:
        st.subheader('Collision severity')
        counts = levels.reindex(SEVERITIES, fill_value=0).reset_index()
        counts.columns = ['Severity', 'Collisions']
        fig = px.bar(counts, x='Severity', y='Collisions', color='Severity',
                     color_discrete_map={'Slight': '#318C82', 'Serious': '#E6A14A', 'Fatal': '#C9535D'},
                     text='Collisions')
        fig.update_traces(texttemplate='%{text:,}', textposition='outside')
        fig.update_layout(showlegend=False, margin=dict(t=20, b=10),
                          template='plotly_dark', paper_bgcolor='#171717',
                          plot_bgcolor='#242424', font=dict(color='#FFFFFF'))
        st.plotly_chart(fig, use_container_width=True)
    with b:
        st.subheader('Speed limits')
        counts = df['speed_limit'].value_counts().sort_index()
        fig = px.bar(x=[f'{x} mph' for x in counts.index], y=counts.values,
                     labels={'x': 'Speed limit', 'y': 'Collisions'})
        fig.update_traces(marker_color='#287B98')
        fig.update_layout(margin=dict(t=20, b=10),
                          template='plotly_dark', paper_bgcolor='#171717',
                          plot_bgcolor='#242424', font=dict(color='#FFFFFF'))
        st.plotly_chart(fig, use_container_width=True)
    st.subheader('Dataset preview — descriptive categories')
    show_cols = ['collision_index', 'date', 'collision_severity', 'weather_conditions',
                 'light_conditions', 'road_surface_conditions', 'road_type', 'speed_limit']
    st.dataframe(df[[c for c in show_cols if c in df]].head(20),
                 use_container_width=True, hide_index=True)
    st.caption('Speed limits and collision counts remain numerical because they are measurements.')

with exploration:
    st.subheader('How conditions relate to recorded collision severity')
    a, b, c = st.columns(3)
    with a:
        factor_name = st.selectbox('Condition to examine', list(PLOTS))
    with b:
        selected = st.selectbox('Severity', ['Serious or Fatal', 'Fatal', 'Serious', 'Slight', 'All'])
    with c:
        metric = st.selectbox('Measure', ['Number of collisions', 'Share within condition (%)'])
    col_name = PLOTS[factor_name]
    selected_mask = (
        df['collision_severity'].isin(['Serious', 'Fatal']) if selected == 'Serious or Fatal' else
        pd.Series(True, index=df.index) if selected == 'All' else
        df['collision_severity'].eq(selected)
    )
    keys = df[col_name].fillna('Not recorded').astype(str)
    if col_name == 'speed_limit':
        keys = keys + ' mph'
    totals = keys.value_counts()
    matched = keys[selected_mask].value_counts()
    data = pd.DataFrame({'Condition': totals.index,
                         'All recorded collisions': totals.values,
                         'Selected collisions': matched.reindex(totals.index, fill_value=0).values})
    data['Share within condition (%)'] = (100 * data['Selected collisions'] /
                                          data['All recorded collisions']).round(2)
    if selected == 'All':
        metric = 'Number of collisions'
    graph_col = ('Selected collisions' if metric == 'Number of collisions'
                 else 'Share within condition (%)')
    data = data.sort_values(graph_col, ascending=False)
    chart = px.bar(data, x='Condition', y=graph_col, color_discrete_sequence=['#47C4CC'])
    chart.update_layout(template='plotly_dark', paper_bgcolor='#171717',
                        plot_bgcolor='#242424', font=dict(color='#FFFFFF'))
    st.plotly_chart(chart, use_container_width=True)
    st.dataframe(data, use_container_width=True, hide_index=True)
    st.caption('Percentages describe severity among *recorded injury collisions* in each condition. '
               'They do not measure the likelihood of an accident, and do not establish causation.')

with quality:
    st.subheader('Data quality checks')
    a, b, c = st.columns(3)
    a.metric('Rows', f'{df.shape[0]:,}')
    b.metric('Columns', f'{df.shape[1]}')
    c.metric('Exact duplicate rows', f'{int(df.duplicated().sum()):,}')
    missing = df.isnull().sum().sort_values(ascending=False)
    st.dataframe(pd.DataFrame({'Column': missing.index, 'Missing values': missing.values,
                               'Missing %': (missing.values / len(df) * 100).round(4)}),
                 hide_index=True, use_container_width=True)
    st.info('Some columns may contain category descriptions such as “Unknown” or '
            '“Data missing or out of range” rather than empty cells. '
            'A few local-authority codes lacked a matching guide entry and were left unchanged.')

with modelling:
    st.subheader('Decision Tree — Slight vs Serious/Fatal')
    st.write('The model uses six road and environmental variables, with an 80% training '
             'and 20% testing split. Text categories are converted using One-Hot Encoding.')
    try:
        model, y_test, pred, n_train, n_test = train_tree()
        from sklearn.metrics import recall_score, f1_score
        a, b, c = st.columns(3)
        a.metric('Test accuracy', f'{accuracy_score(y_test, pred):.2%}')
        b.metric('Serious/Fatal recall', f'{recall_score(y_test, pred):.2%}')
        c.metric('Serious/Fatal F1-score', f'{f1_score(y_test, pred):.2f}')
        st.caption(f'Training records: {n_train:,} · Testing records: {n_test:,}. '
                   'Model depth: 5; minimum leaf size: 100; balanced class weights.')
        left, right = st.columns([1.0, 1.0])
        with left:
            st.markdown('**Confusion matrix**')
            fig, ax = plt.subplots(figsize=(5.5, 4.2), facecolor='white')
            ax.set_facecolor('white')
            ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                                   display_labels=['Slight', 'Serious/Fatal']).plot(ax=ax, cmap='Blues',
                                                                                  colorbar=False, values_format='d')
            fig.tight_layout()
            st.pyplot(fig)
            plt.close(fig)
        with right:
            st.markdown('**Classification report**')
            report = pd.DataFrame(classification_report(
                y_test, pred, target_names=['Slight', 'Serious/Fatal'],
                output_dict=True, zero_division=0)).T.round(3)
            st.dataframe(report, use_container_width=True)
            st.warning('The model misses around half of Serious/Fatal collisions, so it '
                       'is useful for demonstrating classification, not for reliable safety decisions.')
        with st.expander('View simplified Decision Tree diagram'):
            fig, ax = plt.subplots(figsize=(19, 9), facecolor='white')
            ax.set_facecolor('white')
            feature_names = model.named_steps['preprocessor'].get_feature_names_out()
            feature_names = [name.replace('categories__', '').replace('speed__', '')
                             for name in feature_names]
            plot_tree(model.named_steps['classifier'], feature_names=feature_names,
                      class_names=['Slight', 'Serious/Fatal'], max_depth=3,
                      rounded=True, filled=True, fontsize=7, ax=ax)
            fig.tight_layout()
            st.pyplot(fig)
            plt.close(fig)
            st.caption('Shows only the first three levels; the trained tree can reach depth five.')
    except Exception as error:
        st.error(f'The Decision Tree could not be evaluated: {error}')


In [ ]:
# Check that the CarApp file written by the cell above is valid Python
from pathlib import Path
import py_compile
app = Path('/content/CarApp.py')
if not app.exists():
    raise FileNotFoundError('Run the previous CarApp creation cell first.')
py_compile.compile(str(app), doraise=True)
print('✅ CarApp.py has correct indentation and syntax.')

In [ ]:
# 5B. Validate CarApp and start a fresh Streamlit server
import os
import py_compile
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

app_path = '/content/CarApp.py'
py_compile.compile(app_path, doraise=True)
print('✅ CarApp.py syntax is correct.')

# Stop only older Streamlit processes running this particular CarApp.
try:
    import psutil
    for proc in psutil.process_iter(['cmdline']):
        cmd = proc.info.get('cmdline') or []
        if proc.pid != os.getpid() and app_path in cmd and 'streamlit' in cmd:
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except psutil.TimeoutExpired:
                proc.kill()
except ImportError:
    print('psutil not available; if port 8501 is occupied, restart your Colab session.')

health = 'http://127.0.0.1:8501/_stcore/health'
def running():
    try:
        return urllib.request.urlopen(health, timeout=2).status == 200
    except Exception:
        return False

log = open('/content/streamlit.log', 'w', encoding='utf-8')
process = subprocess.Popen([
    sys.executable, '-m', 'streamlit', 'run', app_path,
    '--server.address=127.0.0.1', '--server.port=8501',
    '--server.headless=true', '--browser.gatherUsageStats=false'
], stdout=log, stderr=subprocess.STDOUT)
for _ in range(25):
    if running():
        break
    time.sleep(1)

if running():
    print('✅ CarApp is running (HTTP 200).')
else:
    print('❌ CarApp could not start. Recent log:')
    print(Path('/content/streamlit.log').read_text(encoding='utf-8')[-5000:])


In [ ]:
# 5C. Start a temporary Cloudflare link (does not block the notebook)
import os
import re
import subprocess
import time
from pathlib import Path
import urllib.request

if not running():
    raise RuntimeError('Run step 5B first; Streamlit must be running.')

cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    url = ('https://github.com/cloudflare/cloudflared/releases/latest/download/'
           'cloudflared-linux-amd64')
    print('Downloading Cloudflare tunnel helper...')
    urllib.request.urlretrieve(url, cloudflared)
    cloudflared.chmod(0o755)

# Reuse an existing tunnel if it is still running; otherwise start one
log_path = Path('/content/cloudflared.log')
old_proc = globals().get('tunnel_process')
if old_proc is None or old_proc.poll() is not None:
    log = open(log_path, 'w', encoding='utf-8')
    tunnel_process = subprocess.Popen(
        [str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8501'],
        stdout=log, stderr=subprocess.STDOUT
    )

link = None
for _ in range(30):
    logs = log_path.read_text(errors='replace') if log_path.exists() else ''
    found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', logs)
    if found:
        link = found.group(0)
        break
    time.sleep(1)

if link:
    print('✅ Open CarApp at:', link)
    print('Keep this Colab runtime running while using the website.')
else:
    print('A link has not appeared yet. Check these logs:')
    print(logs[-4000:])

### Interpretation for your report and presentation

- The Decision Tree predicts **Slight** versus **Serious/Fatal** among recorded injury collisions.
- The original labelled 2025 dataset produced **58.12% accuracy** with the specified training split and six variables; rerunning on that same dataset should reproduce it.
- Counts of collisions during weather and lighting conditions are not the same as the risk of having a collision, because the data do not provide traffic-exposure information.
- The Cloudflare website URL is temporary; keep your Colab runtime running while using it.